# SEER MLP benchmark

The model predicts death before 60 months from diagnosis-time features. Patients alive with less than 60 months of follow-up are excluded because their five-year outcome is unknown.

- **Default experiment** uses the compact MLP and nested cross-validation to compare BCE with focal loss. Both losses use the same folds and their own hyperparameter search.
- **Fixed-split workflow** compares the original and regularized networks with and without class weighting. It also supports the optional final test, calibration, fairness and explanation sections.
- **Configure the run** in the Hyperparameters and experiment settings cell. The default stratified split reserves 20% for the final test and uses the other 80% for development.


## 1. Environment and configuration
Python 3.11+ is recommended. In a terminal, install CPU PyTorch with `python -m pip install torch --index-url https://download.pytorch.org/whl/cpu`, then run `python -m pip install -r requirements.txt`. For Windows CUDA, use the command from the [official PyTorch installation selector](https://pytorch.org/get-started/locally/) for your driver/platform **instead of the CPU command**, then install requirements. In Colab, keep the preinstalled CUDA PyTorch; upload the CSV or configure its Drive path. Restart the kernel after installation. This notebook never installs packages automatically.

SHAP uses [GradientExplainer / expected gradients](https://shap.readthedocs.io/en/stable/generated/shap.GradientExplainer.html) on logits. CPU/GPU results need not be bitwise identical; deterministic settings reduce, but cannot eliminate, platform differences.


### Imports

In [1]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import copy
import hashlib
import importlib.metadata as metadata
import json
import platform
import random
import shutil
import uuid
from itertools import product
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.special import expit
from scipy.stats import spearmanr
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, precision_score,
    precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import Markdown, display

### Data paths and feature schema

In [2]:
PROJECT_DIR = Path(r"C:\Users\Tommaso\Code\progetto")
CSV_NAME = "Copia di SEER Breast Cancer Dataset .csv"
DATA_PATH = None  # Optional explicit local/Colab path.
SPLIT_PATH = None  # CSV columns: row_id, split. See section 4.
BASELINE_PATH = None  # CSV: row_id, y_true, probability, model.
BASELINE_SHAP_PATH = None  # Optional grouped logit attribution CSV; see section 13.

NUM_COLS = ["Age", "Tumor Size", "Regional Node Examined", "Reginol Node Positive"]
CAT_COLS = ["Race", "Marital Status", "T Stage", "N Stage", "6th Stage",
            "Grade", "A Stage", "Estrogen Status", "Progesterone Status"]
FEATURES = NUM_COLS + CAT_COLS
# No age strata were defined in the inspected project notebook. Prespecify these:
AGE_BINS = [-np.inf, 50, 60, np.inf]
AGE_LABELS = ["<50", "50-59", ">=60"]

### Hyperparameters and experiment settings
Edit this cell to change the split, training settings or search grid. The learning rates in the grid are starting values for the adaptive scheduler.

In [3]:
# Execution and reproducibility
DEVICE_MODE = "auto"  # auto, cpu, cuda
USE_AMP = False
DETERMINISTIC = True
SPLIT_SEED = 42
SEEDS = [42, 123, 2026]
# Dataset partitions and experiment mode
TEST_SIZE = 0.20
VALIDATION_FRACTION = 0.25  # Of the baseline training pool.
REPRODUCE_PROJECT_SPLIT = False  # Set True with TEST_SIZE=0.25 for the historical baseline split.
# Keep False during network development: Run All stops at validation diagnostics.
RUN_FINAL_EVALUATION = False
RUN_NESTED_LOSS_CV = True  # False restores the original fixed-split workflow.
# Nested CV and the BCE / focal search grid
CV_OUTER_FOLDS = 5
CV_INNER_FOLDS = 3
CV_INNER_SEED = 42
CV_STOP_FRACTION = 0.15
CV_LEARNING_RATES = (0.0005, 0.001)
CV_DROPOUTS = (0.1, 0.2)
CV_WEIGHT_DECAYS = (0.0001, 0.001)
CV_FOCAL_GAMMAS = (1.0, 2.0)
# Final evaluation and reporting
POSTHOC_CALIBRATION = True
RUN_SHAP = True
SHAP_BACKGROUND = 64
SHAP_PATIENTS = 100
SHAP_SAMPLES = 200
BOOTSTRAPS = 300
MIN_GROUP_N = 30
MIN_GROUP_CLASS = 10
# Architecture, optimizer, scheduler and early stopping
HP = dict(hidden=[32, 16], dropout=0.1, learning_rate=0.001,
          batch_size=32, max_epochs=200, patience=20, weight_decay=0.0001,
          gradient_clip=1.0, min_delta=1e-5, scheduler=False,
          scheduler_patience=4, scheduler_factor=0.5, minimum_lr=1e-5)
# Deliberately small development comparison. The original settings remain a comparator.
NETWORK_CANDIDATES = {
    "original": dict(HP),
    "regularized": dict(HP, dropout=0.2, weight_decay=0.001,
                        learning_rate=0.0005, scheduler=True),
    "compact_regularized": dict(HP, hidden=[16, 8], dropout=0.2,
                        weight_decay=0.001, learning_rate=0.0005, scheduler=True),
}
LOG_EVERY = 5

### Device and reproducibility

In [4]:
if RUN_NESTED_LOSS_CV and RUN_FINAL_EVALUATION:
    raise ValueError("Final test evaluation belongs to the fixed-split workflow. Disable it for nested loss CV.")
assert DEVICE_MODE in {"auto", "cpu", "cuda"}
assert len(set(SEEDS)) >= 3
if DEVICE_MODE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but unavailable. Install a CUDA build/check driver.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if DEVICE_MODE == "auto" else torch.device(DEVICE_MODE)
AMP_ENABLED = USE_AMP and device.type == "cuda"
if USE_AMP and not AMP_ENABLED:
    warnings.warn("AMP disabled because the selected device is CPU.")

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(DETERMINISTIC, warn_only=True)
    torch.backends.cudnn.deterministic = DETERMINISTIC
    torch.backends.cudnn.benchmark = not DETERMINISTIC

seed_everything(SPLIT_SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
VERSIONS = {name: metadata.version(name) for name in
            ["torch", "numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "joblib", "shap"]}
ENVIRONMENT = dict(python=platform.python_version(), packages=VERSIONS,
    cuda_available=torch.cuda.is_available(), cuda_runtime=torch.version.cuda,
    selected_device=str(device), amp=AMP_ENABLED)
if device.type == "cuda":
    free, total = torch.cuda.mem_get_info(device)
    ENVIRONMENT.update(gpu=torch.cuda.get_device_name(device),
                       free_vram_gib=free / 2**30, total_vram_gib=total / 2**30)
display(ENVIRONMENT)
plt.rcParams.update({"figure.figsize": (8, 4.8), "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})


{'python': '3.11.18',
 'packages': {'torch': '2.15.0+cpu',
  'numpy': '2.4.6',
  'pandas': '2.3.3',
  'scikit-learn': '1.9.1',
  'scipy': '1.17.1',
  'matplotlib': '3.11.2',
  'joblib': '1.6.0',
  'shap': '0.51.0'},
 'cuda_available': False,
 'cuda_runtime': None,
 'selected_device': 'cpu',
 'amp': False}

## 2. Run directory and tracking
Every execution gets a UTC timestamp plus random suffix, preventing overwrites. The dataset hash, cleaned-to-original row mapping and split manifest travel with every run. Checkpoints use CPU tensors and can be loaded with `map_location=device`; load only trusted checkpoints/preprocessors.


In [5]:
OUTPUT_ROOT = (PROJECT_DIR if PROJECT_DIR.is_dir() else Path.cwd()) / "results" / "mlp"
RUN_DIR = OUTPUT_ROOT / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
for folder in ["checkpoints", "figures", "metrics", "predictions", "explainability", "fairness", "configs"]:
    (RUN_DIR / folder).mkdir(parents=True, exist_ok=False)

def save_json(value, path):
    path.write_text(json.dumps(value, indent=2, default=str, allow_nan=False), encoding="utf-8")

def save_figure(name):
    plt.gcf().savefig(RUN_DIR / "figures" / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()

save_json(ENVIRONMENT, RUN_DIR / "configs" / "environment.json")
CONFIG = dict(hyperparameters=HP, network_candidates=NETWORK_CANDIDATES,
    final_evaluation=RUN_FINAL_EVALUATION,
    network_selection="mean stopping-validation AP across both weighting strategies and all fixed seeds", seeds=SEEDS, split_seed=SPLIT_SEED,
    device_mode=DEVICE_MODE, deterministic=DETERMINISTIC, amp=USE_AMP,
    age_bins=["-inf", 50, 60, "inf"], age_labels=AGE_LABELS,
    validation_fraction=VALIDATION_FRACTION, test_size=TEST_SIZE,
    posthoc_calibration=POSTHOC_CALIBRATION, split_path=str(SPLIT_PATH),
    baseline_path=str(BASELINE_PATH), numerical=NUM_COLS, categorical=CAT_COLS,
    bootstrap_replicates=BOOTSTRAPS, min_group_n=MIN_GROUP_N,
    min_group_class=MIN_GROUP_CLASS,
    shap=dict(enabled=RUN_SHAP, background=SHAP_BACKGROUND, patients=SHAP_PATIENTS, samples=SHAP_SAMPLES),
    threshold_criterion="maximum validation balanced accuracy; ties nearest 0.5")
print("Outputs:", RUN_DIR)


Outputs: C:\Users\Tommaso\Code\progetto\results\mlp\20261111T121452Z_ca6e014c


## 3. Cohort, outcome and identifiers
Match the existing cleaning exactly: drop the empty column, strip column names, retain the first full-record duplicate, then reset cleaned indices. `row_id` always denotes the **zero-based original CSV data row**, excluding the header; `clean_id` denotes the post-deduplication index. The baseline builds its task cohort by concatenating survivors first and early deaths second; preserve that order when reproducing its split.

The expected eligible counts are 3,270 / 2,821 / 449. Deviations stop execution rather than silently change the protocol. Death at exactly 60 months belongs to class 0. Excluding early censoring does not correct informative censoring. No outliers are removed. Race is a recorded dataset category, not a biological explanation.


In [6]:
paths = ([Path(DATA_PATH)] if DATA_PATH else []) + [PROJECT_DIR / CSV_NAME, Path.cwd() / CSV_NAME, Path("/content") / CSV_NAME]
dataset_path = next((p for p in paths if p.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError("CSV unavailable. Set DATA_PATH, upload to /content, or mount Drive and set DATA_PATH.")
raw = pd.read_csv(dataset_path)
raw.columns = raw.columns.str.strip()
raw = raw.drop(columns=["Unnamed: 3"], errors="ignore")
assert set(FEATURES + ["Status", "Survival Months"]).issubset(raw.columns)
assert raw[FEATURES + ["Status", "Survival Months"]].notna().all().all()
duplicate_mask = raw.duplicated(keep="first")
df = raw.loc[~duplicate_mask].copy()
df["row_id"] = df.index.astype(int)
df = df.reset_index(drop=True)
df["clean_id"] = df.index
assert set(df.Status.unique()) <= {"Alive", "Dead"}, "Unexpected status: inspect protocol."
assert (df["Survival Months"] >= 0).all()
survived = df.loc[df["Survival Months"] >= 60].assign(label=0)
died_early = df.loc[(df.Status == "Dead") & (df["Survival Months"] < 60)].assign(label=1)
cohort = pd.concat([survived, died_early], ignore_index=True).set_index("row_id", drop=False)
excluded = df.loc[(df.Status == "Alive") & (df["Survival Months"] < 60)]
observed = (len(cohort), int((cohort.label == 0).sum()), int(cohort.label.sum()))
if observed != (3270, 2821, 449):
    raise ValueError(f"Cohort differs from existing protocol: {observed}; inspect source before proceeding.")
assert len(cohort) + len(excluded) == len(df)
assert (cohort.loc[cohort.label == 1, "Survival Months"] < 60).all()
AUDIT = dict(raw_records=len(raw), exact_duplicates_removed=int(duplicate_mask.sum()),
    eligible=len(cohort), class_0=observed[1], class_1=observed[2],
    censored_excluded=len(excluded), late_deaths_class_0=int((survived.Status == "Dead").sum()),
    dataset_sha256=hashlib.sha256(dataset_path.read_bytes()).hexdigest())
display(AUDIT)
save_json(AUDIT, RUN_DIR / "configs" / "cohort_audit.json")
df[["row_id", "clean_id"]].to_csv(RUN_DIR / "configs" / "row_mapping.csv", index=False)
# Report structural anomalies; do not silently delete/repair clinical data.
anomalies = cohort.loc[(cohort["Reginol Node Positive"] > cohort["Regional Node Examined"]) |
                      (cohort[NUM_COLS] < 0).any(axis=1), ["row_id"] + NUM_COLS]
anomalies.to_csv(RUN_DIR / "metrics" / "structural_anomalies.csv", index=False)
if len(anomalies):
    warnings.warn(f"{len(anomalies)} structural anomalies retained; excluded from perturbation experiments.")


{'raw_records': 4024,
 'exact_duplicates_removed': 1,
 'eligible': 3270,
 'class_0': 2821,
 'class_1': 449,
 'censored_excluded': 753,
 'late_deaths_class_0': 167,
 'dataset_sha256': '1d64363e87b306bdb241cb7e105271514f74564842c02e01dbce2f4884266d09'}

## 4. Fixed partitions and baseline compatibility
An imported split CSV must contain `row_id,split`, with `split` in `train,validation,test` (or `train,test` only), and exactly one row per eligible record. Convert baseline task indices using the original-row mapping first: do not assume task index equals CSV row ID. The validation pool is internal to baseline training; half is used for early stopping, one quarter for threshold tuning, one quarter for optional calibration. These small validation samples limit precision; parameters are prespecified, no architecture search is performed.

Without an imported manifest, the inspected `breast_cancer.ipynb` split can be reproduced using its survivor/death ordering, deduplication, `test_size=0.25` and `random_state=42`. This reproduces that source recipe, not an independently verified external run; baseline imports still enforce exact patient alignment. Disable `REPRODUCE_PROJECT_SPLIT` for a generic stratified fallback with a comparability warning. No verified patient identifier exists; full-record duplicates are removed, but repeated visits with different records cannot be ruled out.


The default MLP split is stratified 80% development and 20% held-out test. To reproduce the historical split, set `TEST_SIZE=0.25`, `SPLIT_SEED=42` and `REPRODUCE_PROJECT_SPLIT=True`. A supplied split manifest takes precedence.


In [7]:
all_ids = cohort.row_id.to_numpy()
y_all = cohort.label
if SPLIT_PATH:
    supplied = pd.read_csv(SPLIT_PATH)
    assert {"row_id", "split"}.issubset(supplied.columns)
    assert not supplied.row_id.duplicated().any()
    assert set(supplied.row_id) == set(all_ids), "Manifest must cover the exact eligible cohort."
    assert set(supplied.split) <= {"train", "validation", "test"}
    parts = {part: supplied.loc[supplied.split == part, "row_id"].to_numpy() for part in ["train", "validation", "test"]}
    train_ids, val_ids, test_ids = parts["train"], parts["validation"], parts["test"]
    if not len(val_ids):
        train_ids, val_ids = train_test_split(train_ids, test_size=VALIDATION_FRACTION,
            stratify=y_all.loc[train_ids], random_state=SPLIT_SEED)
    split_origin = "imported original-row manifest"
else:
    if REPRODUCE_PROJECT_SPLIT:
        assert SPLIT_SEED == 42 and TEST_SIZE == 0.25, "Project source uses seed 42 and test_size 0.25."
        split_origin = "reproduced breast_cancer.ipynb source recipe; verify baseline imports"
    else:
        warnings.warn("Custom stratified split: baseline comparisons require an exact aligned test set.")
        split_origin = "custom stratified split; comparability requires aligned test IDs"
    train_pool, test_ids = train_test_split(all_ids, test_size=TEST_SIZE,
        stratify=y_all.loc[all_ids], random_state=SPLIT_SEED)
    train_ids, val_ids = train_test_split(train_pool, test_size=VALIDATION_FRACTION,
        stratify=y_all.loc[train_pool], random_state=SPLIT_SEED)
stop_ids, auxiliary_ids = train_test_split(val_ids, test_size=0.5,
    stratify=y_all.loc[val_ids], random_state=SPLIT_SEED)
tune_ids, cal_ids = train_test_split(auxiliary_ids, test_size=0.5,
    stratify=y_all.loc[auxiliary_ids], random_state=SPLIT_SEED)
PARTS = dict(train=train_ids, validation_stop=stop_ids, validation_threshold=tune_ids,
             validation_calibration=cal_ids, test=test_ids)
manifest = pd.concat([pd.DataFrame({"row_id": ids, "split": part}) for part, ids in PARTS.items()], ignore_index=True)
assert not manifest.row_id.duplicated().any()
assert set(manifest.row_id) == set(all_ids)
for part, ids in PARTS.items():
    assert len(ids) and y_all.loc[ids].nunique() == 2, f"Both classes required in {part}."
# Guard exact clinical-record leakage independently of manifest identity.
record_hash = pd.util.hash_pandas_object(cohort[FEATURES + ["Status", "Survival Months"]], index=False)
hash_partition = manifest.assign(record_hash=manifest.row_id.map(record_hash))
assert hash_partition.groupby("record_hash").split.nunique().max() == 1
manifest = manifest.merge(cohort[["row_id", "clean_id", "label"]].reset_index(drop=True), on="row_id", validate="one_to_one")
manifest.to_csv(RUN_DIR / "configs" / "split_manifest.csv", index=False)
CONFIG.update(split_origin=split_origin, dataset_sha256=AUDIT["dataset_sha256"])
save_json(CONFIG, RUN_DIR / "configs" / "experiment.json")
display(manifest.groupby("split").label.agg(patients="size", positives="sum", prevalence="mean"))


C:\Users\Tommaso\AppData\Local\Temp\ipykernel_25496\3428694364.py:20: UserWarning: Custom stratified split: baseline comparisons require an exact aligned test set.
  warnings.warn("Custom stratified split: baseline comparisons require an exact aligned test set.")


,patients,positives,prevalence
split,,,
test,654,90,0.137615
train,1962,269,0.137105
validation_calibration,164,23,0.140244
validation_stop,327,45,0.137615
validation_threshold,163,22,0.134969


## 5. Training-only preprocessing and augmentation interface
Standardize all four numeric features; one-hot encode all nine categoricals (including clinical stages) to avoid invented ordinal distances. This differs from baseline feature engineering, so model comparisons assess entire pipelines. Categories absent from training map to an all-zero group and are audited. The input dimension is inferred, not fixed to 34. Preprocessing is fitted once on original training patients only and reused for all seeds/configurations.

`prepare_training_data(synthetic_features, synthetic_labels)` accepts externally validated minority records in **original feature space**. It does not generate data or refit preprocessing. Future studies must verify generator training provenance, clinical consistency, overlap and minority labels, and keep this manifest and preprocessor fixed. This experiment calls the function without synthetic records. It rejects class-weight-plus-augmentation combinations by default.


In [8]:
preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), NUM_COLS),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), CAT_COLS),
], remainder="drop", verbose_feature_names_out=False)

if not RUN_NESTED_LOSS_CV:
    X_train = cohort.loc[train_ids, FEATURES].copy()
    y_train = y_all.loc[train_ids].to_numpy(dtype=np.float32)
    preprocessor.fit(X_train)
    joblib.dump(preprocessor, RUN_DIR / "checkpoints" / "preprocessor.joblib")
    encoded_names = preprocessor.get_feature_names_out().tolist()
    encoded_groups = NUM_COLS.copy()
    for col, categories in zip(CAT_COLS, preprocessor.named_transformers_["categorical"].categories_):
        encoded_groups.extend([col] * len(categories))
    assert len(encoded_names) == len(encoded_groups)
    pd.DataFrame({"encoded_feature": encoded_names, "original_feature": encoded_groups}).to_csv(
        RUN_DIR / "configs" / "feature_mapping.csv", index=False)

    def transform(frame):
        values = preprocessor.transform(frame[FEATURES]).astype(np.float32)
        assert np.isfinite(values).all()
        return values

    def prepare_training_data(synthetic_features=None, synthetic_labels=None):
        base_x, base_y = transform(X_train), y_train.copy()
        if synthetic_features is None:
            assert synthetic_labels is None
            return base_x, base_y
        assert synthetic_labels is not None
        assert set(synthetic_features.columns) == set(FEATURES)
        sy = np.asarray(synthetic_labels, dtype=np.float32)
        assert sy.ndim == 1 and len(sy) == len(synthetic_features) and (sy == 1).all()
        assert synthetic_features.notna().all().all()
        assert (synthetic_features[NUM_COLS] >= 0).all().all()
        assert (synthetic_features["Reginol Node Positive"] <= synthetic_features["Regional Node Examined"]).all()
        for col in CAT_COLS:
            assert synthetic_features[col].isin(X_train[col].unique()).all()
        return np.concatenate([base_x, transform(synthetic_features)]), np.concatenate([base_y, sy])

    train_array, train_labels = prepare_training_data()
    stop_array = transform(cohort.loc[stop_ids])
    stop_labels = y_all.loc[stop_ids].to_numpy(dtype=np.float32)
    tune_array = transform(cohort.loc[tune_ids])
    cal_array = transform(cohort.loc[cal_ids])
    INPUT_DIM = train_array.shape[1]
    print("Encoded input dimensions:", INPUT_DIM)
    unknowns = []
    for part, ids in PARTS.items():
        for col in CAT_COLS:
            unknowns.append(dict(split=part, feature=col,
                unknown_count=int((~cohort.loc[ids, col].isin(X_train[col].unique())).sum())))
    pd.DataFrame(unknowns).to_csv(RUN_DIR / "metrics" / "unknown_categories.csv", index=False)


## 6. MLP and fixed-split development
Compare three prespecified candidates across the same three seeds and both weighting strategies: original [32,16], regularized [32,16], compact regularized [16,8]. The two new candidates increase dropout to 0.2 and AdamW weight decay to 0.001, halve the initial learning rate, and reduce it on a validation-loss plateau. These are candidate bundles, not isolated causal ablations. The original candidate retains the original optimization settings.

Within each run, restore the **strictly lowest validation objective**. `min_delta` controls patience resets independently, avoiding loss of a slightly better checkpoint. Weighted and unweighted BCE magnitudes must not be compared directly. Validation AP, ROC-AUC, unweighted BCE and Brier are also logged. Measure training loss again in eval mode after each epoch so the train/validation gap compares the same frozen parameters with dropout disabled; the online training loss remains available separately.

Choose one common candidate for A/B by mean validation AP across all six runs (ties: lower AP SD, then fewer parameters); report both weighting families individually as well. This keeps their architecture/hyperparameters matched. Reusing the stopping-validation set for this bounded development choice yields optimistic development scores; it is not independent validation. The threshold/calibration subsets remain reserved. Larger searches would need training-only nested validation/CV. The old 42-event stopping set limits selection precision, so differences across seeds are descriptive.

The scheduler reduces the learning rate when validation loss plateaus ([PyTorch documentation](https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.ReduceLROnPlateau.html)); early stopping still ends runs without useful improvement. More epochs alone are not a remedy for the observed divergence. No resampling or synthetic data is introduced.


### Model, batches and prediction

In [9]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden=None, dropout=0.1):
        super().__init__()
        layers, previous = [], input_dim
        for width in (hidden or [32, 16]):
            layers.extend([nn.Linear(previous, width), nn.ReLU(), nn.Dropout(dropout)])
            previous = width
        layers.append(nn.Linear(previous, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)  # Shape (batch, 1); explainers also use this shape.

def loader(x, y, shuffle=False, seed=42, batch_size=None):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(TensorDataset(torch.from_numpy(np.asarray(x, dtype=np.float32)),
                                   torch.from_numpy(np.asarray(y, dtype=np.float32))),
        batch_size=batch_size or HP["batch_size"], shuffle=shuffle, num_workers=0,
        pin_memory=device.type == "cuda", generator=generator)

def predict_logits(model, x):
    model.eval()
    pieces = []
    with torch.no_grad():
        for start in range(0, len(x), 512):
            batch = torch.from_numpy(x[start:start + 512]).to(device)
            pieces.append(model(batch).squeeze(1).float().cpu().numpy())
    return np.concatenate(pieces)

### BCE and focal objectives

In [10]:
def objective_from_logits(logits, labels, positive_weight, gamma=0.0):
    # Stable numpy BCE; also allows comparable unweighted BCE for weighted models.
    logits = np.asarray(logits, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)
    signed_logits = (2 * labels - 1) * logits
    bce = np.logaddexp(0, -signed_logits)
    factor = (-np.expm1(-bce)) ** gamma if gamma else 1.0
    return float(np.mean(bce * factor * np.where(labels == 1, positive_weight, 1.0)))

def focal_loss_from_logits(logits, labels, gamma):
    bce = nn.functional.binary_cross_entropy_with_logits(logits.float(), labels, reduction="none")
    if gamma == 0:
        return bce.mean()
    # 1 - p_t, computed accurately even for easy examples.
    return (bce * (-torch.expm1(-bce)).pow(gamma)).mean()

### Training loop and early stopping

In [11]:
def fit_model(x, y, validation_x, validation_y, *, weighted, seed, tag,
              augmented=False, hyperparameters=None, save_artifacts=True,
              feature_names=None, verbose=True):
    hp = copy.deepcopy(hyperparameters if hyperparameters is not None else HP)
    gamma = float(hp.get("focal_gamma", 0.0))
    if not np.isfinite(gamma) or gamma < 0 or (gamma and weighted):
        raise ValueError("Use a finite nonnegative focal gamma and no class weighting with focal loss.")
    x, y = np.asarray(x, dtype=np.float32), np.asarray(y, dtype=np.float32)
    validation_x = np.asarray(validation_x, dtype=np.float32)
    validation_y = np.asarray(validation_y, dtype=np.float32)
    if weighted and augmented:
        raise ValueError("Class weighting plus augmentation requires a separate protocol.")
    if set(np.unique(y)) != {0, 1} or set(np.unique(validation_y)) != {0, 1}:
        raise ValueError("Training and stopping validation must each contain both classes.")
    if not np.isfinite(x).all() or not np.isfinite(validation_x).all():
        raise ValueError("Nonfinite features.")
    seed_everything(seed)
    model = MLP(x.shape[1], hp["hidden"], hp["dropout"]).to(device)
    weight = float((y == 0).sum() / (y == 1).sum()) if weighted else 1.0
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weight, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=hp["learning_rate"], weight_decay=hp["weight_decay"])
    scheduler = (torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
        factor=hp["scheduler_factor"], patience=hp["scheduler_patience"],
        threshold=hp["min_delta"], threshold_mode="abs", min_lr=hp["minimum_lr"])
        if hp["scheduler"] else None)
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    else:
        scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
    train_loader = loader(x, y, shuffle=True, seed=seed, batch_size=hp["batch_size"])
    best_loss, patience_anchor, stale = np.inf, np.inf, 0
    best_state, best_epoch = None, 0
    history = []
    for epoch in range(1, hp["max_epochs"] + 1):
        learning_rate = float(optimizer.param_groups[0]["lr"])
        model.train()
        training_total = 0.0
        for bx, by in train_loader:
            bx = bx.to(device, non_blocking=device.type == "cuda")
            by = by.to(device, non_blocking=device.type == "cuda")
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=AMP_ENABLED):
                logits = model(bx).squeeze(1)
                loss = focal_loss_from_logits(logits, by, gamma) if gamma else criterion(logits, by)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Nonfinite training loss: {tag}, epoch {epoch}")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), hp["gradient_clip"], error_if_nonfinite=True)
            scaler.step(optimizer)
            scaler.update()
            training_total += loss.detach().item() * len(by)
        train_logits = predict_logits(model, x)
        val_logits = predict_logits(model, validation_x)
        val_probability = expit(val_logits)
        train_eval_loss = objective_from_logits(train_logits, y, weight, gamma)
        val_loss = objective_from_logits(val_logits, validation_y, weight, gamma)
        if not np.isfinite([train_eval_loss, val_loss]).all():
            raise FloatingPointError(f"Nonfinite evaluation objective in {tag}.")
        history.append(dict(epoch=epoch, training_loss=training_total/len(y),
            train_eval_loss=train_eval_loss, validation_loss=val_loss,
            validation_bce=objective_from_logits(val_logits, validation_y, 1.0),
            validation_ap=average_precision_score(validation_y, val_probability),
            validation_roc_auc=roc_auc_score(validation_y, val_probability),
            validation_brier=brier_score_loss(validation_y, val_probability),
            learning_rate=learning_rate))
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        if val_loss < patience_anchor - hp["min_delta"]:
            patience_anchor, stale = val_loss, 0
        else:
            stale += 1
        if scheduler is not None:
            scheduler.step(val_loss)
        if verbose and (epoch == 1 or epoch % LOG_EVERY == 0 or stale >= hp["patience"]):
            print(f"{tag} | epoch {epoch:3d} | train(eval) {train_eval_loss:.4f} | "
                  f"val {val_loss:.4f} | val AP {history[-1]['validation_ap']:.3f} | "
                  f"lr {learning_rate:.2g} | best {best_epoch}", flush=True)
        if stale >= hp["patience"]:
            break
    assert best_state is not None
    model.load_state_dict(best_state)
    history = pd.DataFrame(history)
    if save_artifacts:
        torch.save(dict(state_dict=best_state, input_dim=x.shape[1], hidden=hp["hidden"],
            dropout=hp["dropout"], seed=seed, weighted=weighted, pos_weight=weight,
            best_epoch=best_epoch, features=encoded_names if feature_names is None else feature_names, hyperparameters=hp,
            parameter_count=sum(p.numel() for p in model.parameters())),
            RUN_DIR / "checkpoints" / f"{tag}.pt")
        history.to_csv(RUN_DIR / "metrics" / f"history_{tag}.csv", index=False)
    return model, history, best_epoch

### Threshold selection for the fixed-split workflow

In [12]:
def choose_threshold(labels, probability):
    candidates = np.unique(np.r_[0.0, probability, 0.5, 1.0])
    scores = np.array([balanced_accuracy_score(labels, probability >= t) for t in candidates])
    tied = np.flatnonzero(np.isclose(scores, scores.max()))
    winner = tied[np.argmin(np.abs(candidates[tied] - 0.5))]
    return float(candidates[winner])

### Run the fixed-split comparison

In [13]:
if not RUN_NESTED_LOSS_CV:
    # All candidate selection uses stopping-validation labels only.
    candidate_runs, development_rows = {}, []
    for candidate, hp in NETWORK_CANDIDATES.items():
        for weighted, name in [(False, "MLP-A"), (True, "MLP-B")]:
            for seed in SEEDS:
                tag = f"{candidate}_{name}_seed{seed}"
                model, history, best_epoch = fit_model(train_array, train_labels,
                    stop_array, stop_labels, weighted=weighted, seed=seed, tag=tag,
                    hyperparameters=hp)
                best = history.loc[history.epoch == best_epoch].iloc[0]
                parameter_count = sum(p.numel() for p in model.parameters())
                development_rows.append(dict(candidate=candidate, model=name, seed=seed,
                    parameters=parameter_count, best_epoch=best_epoch, epochs_run=len(history),
                    validation_ap=float(best.validation_ap), validation_loss=float(best.validation_loss),
                    validation_bce=float(best.validation_bce), validation_brier=float(best.validation_brier),
                    objective_gap=float(best.validation_loss-best.train_eval_loss)))
                # Keep candidate models on CPU between runs; selected models return to device below.
                model.to("cpu")
                candidate_runs[(candidate, name, seed)] = dict(model=model, history=history,
                    best_epoch=best_epoch, checkpoint_tag=tag)
    development = pd.DataFrame(development_rows)
    development.to_csv(RUN_DIR / "metrics" / "network_development.csv", index=False)
    development.groupby(["candidate", "model"])[["validation_ap", "validation_bce", "validation_brier", "objective_gap"]].agg(["mean", "std"]).to_csv(
        RUN_DIR / "metrics" / "network_development_by_weighting.csv")
    candidate_summary = development.groupby("candidate").agg(mean_validation_ap=("validation_ap", "mean"),
        sd_validation_ap=("validation_ap", "std"), parameters=("parameters", "first"))
    candidate_summary = candidate_summary.sort_values(["mean_validation_ap", "sd_validation_ap", "parameters"], ascending=[False, True, True])
    candidate_summary.to_csv(RUN_DIR / "metrics" / "network_candidate_summary.csv")
    selected_candidate = str(candidate_summary.index[0])
    selected_hp = copy.deepcopy(NETWORK_CANDIDATES[selected_candidate])
    HP = copy.deepcopy(selected_hp)  # Future augmentation reuses the selected classifier by default.
    CONFIG.update(hyperparameters=HP, selected_candidate=selected_candidate, selected_hyperparameters=selected_hp)
    save_json(CONFIG, RUN_DIR / "configs" / "experiment.json")
    display(candidate_summary.round(4))
    print("Common candidate selected on development validation only:", selected_candidate)

    experiments = {}
    for name in ["MLP-A", "MLP-B"]:
        for seed in SEEDS:
            selected = candidate_runs[(selected_candidate, name, seed)]
            model = selected["model"].to(device)
            tag = f"{name}_seed{seed}"
            threshold, calibrator, calibrated_threshold = None, None, None
            if RUN_FINAL_EVALUATION:
                tune_logits = predict_logits(model, tune_array)
                threshold = choose_threshold(y_all.loc[tune_ids], expit(tune_logits))
                if POSTHOC_CALIBRATION:
                    calibrator = LogisticRegression(C=1.0, solver="lbfgs", random_state=SPLIT_SEED)
                    calibrator.fit(predict_logits(model, cal_array).reshape(-1, 1), y_all.loc[cal_ids])
                    calibrated_threshold = choose_threshold(y_all.loc[tune_ids],
                        calibrator.predict_proba(tune_logits.reshape(-1, 1))[:, 1])
                    joblib.dump(calibrator, RUN_DIR / "checkpoints" / f"calibrator_{tag}.joblib")
            shutil.copyfile(RUN_DIR / "checkpoints" / f"{selected['checkpoint_tag']}.pt",
                            RUN_DIR / "checkpoints" / f"{tag}.pt")
            experiments[tag] = dict(model=model, name=name, seed=seed, threshold=threshold,
                history=selected["history"], best_epoch=selected["best_epoch"], calibrator=calibrator,
                calibrated_threshold=calibrated_threshold, candidate=selected_candidate,
                hyperparameters=selected_hp)
    save_json({tag: {key: run[key] for key in ["name", "seed", "threshold", "calibrated_threshold", "best_epoch", "candidate", "hyperparameters"]}
               for tag, run in experiments.items()}, RUN_DIR / "configs" / "selected_models.json")


## 7. Development diagnostics and handoff
The candidate figure compares AP on the stopping-validation set, with seed SD (not a confidence interval); inspect A and B separately. Selected-model loss curves compare **eval-mode train loss** and validation loss with identical objective and frozen parameters. They show the true objective gap more clearly than the online dropout-affected loss. Dashed vertical lines indicate restored epochs. AP and learning-rate curves support optimization diagnosis; the stopping rule remains validation objective loss.

The generated `network_development.md` records the development choice without using test labels. Keep `RUN_FINAL_EVALUATION=False` while reviewing these diagnostics. Freeze the candidate list and selection rule before enabling the final test section; do not use test results to choose a different candidate.


In [14]:
if not RUN_NESTED_LOSS_CV:
    display(development.groupby(["candidate", "model"])[
        ["validation_ap", "validation_bce", "validation_brier", "best_epoch", "objective_gap"]].agg(["mean", "std"]).round(4))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
    for ax, name in zip(axes, ["MLP-A", "MLP-B"]):
        subset = development[development.model == name]
        grouped = subset.groupby("candidate").validation_ap.agg(["mean", "std"]).reindex(NETWORK_CANDIDATES)
        ax.bar(range(len(grouped)), grouped["mean"], yerr=grouped["std"], capsize=4, color="#27678b")
        ax.set_xticks(range(len(grouped)), grouped.index, rotation=15, ha="right")
        ax.set(title=f"{name}: development AP ± seed SD", ylabel="Stopping-validation average precision", ylim=(0, 1))
        ax.axhline(float(stop_labels.mean()), linestyle=":", color="grey", label="Stopping-validation prevalence")
        ax.legend(fontsize=8)
    save_figure("network_candidate_validation_comparison")

    fig, axes = plt.subplots(2, 2, figsize=(13, 9), layout="constrained")
    for column, name in enumerate(["MLP-A", "MLP-B"]):
        for run in experiments.values():
            if run["name"] != name:
                continue
            h = run["history"]
            line, = axes[0, column].plot(h.epoch, h.train_eval_loss, label=f"seed {run['seed']} train(eval)")
            color = line.get_color()
            axes[0, column].plot(h.epoch, h.validation_loss, color=color, linestyle="--", label=f"seed {run['seed']} validation")
            axes[0, column].axvline(run["best_epoch"], color=color, linestyle=":", alpha=0.6)
            axes[1, column].plot(h.epoch, h.validation_ap, color=color, label=f"seed {run['seed']}")
        axes[0, column].set(title=f"{name}: {selected_candidate}", xlabel="Epoch", ylabel="Objective-specific BCE")
        axes[1, column].set(title=f"{name}: validation ranking", xlabel="Epoch", ylabel="Validation AP", ylim=(0, 1))
        axes[0, column].legend(fontsize=7)
        axes[1, column].legend(fontsize=8)
    save_figure("selected_network_training_diagnostics")
    fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
    for tag, run in experiments.items():
        ax.step(run["history"].epoch, run["history"].learning_rate, where="post", label=tag)
    ax.set(xlabel="Epoch", ylabel="Learning rate", title="Selected networks: learning-rate schedules", yscale="log")
    ax.legend(fontsize=8)
    save_figure("selected_network_learning_rates")
    lines = ["# MLP network development (no test-based selection)",
             f"Common selected candidate: {selected_candidate}.",
             "Selection: mean stopping-validation AP over both weighting configurations and all fixed seeds. Scores are development estimates, not independent validation.",
             "Candidate comparison: network_candidate_summary.csv; per-weighting/per-seed results: network_development.csv."]
    for candidate, row in candidate_summary.iterrows():
        lines.append(f"{candidate}: mean AP={row.mean_validation_ap:.4f}, SD over seed/weighting combinations={row.sd_validation_ap:.4f}, parameters={int(row.parameters)}.")
    lines.append("Per-weighting plots use seed-only SD. Threshold/calibration subsets remain reserved until final evaluation is enabled. Inspect validation loss, AP, Brier and initialization variability together; selection by AP does not certify fairness or calibration.")
    text = "\n\n".join(lines)
    (RUN_DIR / "network_development.md").write_text(text, encoding="utf-8")
    display(Markdown(text))


## 8. Nested cross-validation for loss and hyperparameters
The compact MLP uses binary focal loss with factor `(1 - p_t)**gamma` multiplying the per-patient BCE, where `p_t` is the probability assigned to the observed class. BCE is the `gamma=0` case. Focal trials omit additional class weights so the comparison tests the focusing term. [Lin et al., ICCV 2017](https://openaccess.thecvf.com/content_iccv_2017/html/Lin_Focal_Loss_for_ICCV_2017_paper.html)

Each outer fold tunes BCE and focal separately using the same inner folds and optimizer grid. Within each inner training partition, a stratified stopping subset selects the checkpoint. The scaler and encoder are fitted on the gradient-training rows. The inner scoring subset ranks configurations by mean AP. For outer evaluation, each selected configuration uses a fresh fitting/stopping split and three matched seeds. The outer prediction is the mean of their sigmoid scores. The outer fold stays outside fitting, early stopping and tuning.

The default grid has 8 BCE and 16 focal configurations. Each configuration uses one common seed during inner tuning. The outer comparison uses three seeds, for 390 model fits total. Every candidate uses the same epoch limit and patience. Raw sigmoid scores may differ in calibration across objectives, so compare both AP and Brier. Fold-level differences are paired and descriptive because training sets overlap. Each fold selects its own configuration. The outer scores evaluate the tuning procedures. A subsequent locked test experiment needs a configuration tuned on the full development pool.


### Search grid and development pool

In [15]:
if RUN_NESTED_LOSS_CV:
    pool_ids = np.concatenate([train_ids, stop_ids, tune_ids, cal_ids])
    pool_labels = y_all.loc[pool_ids].to_numpy(dtype=np.float32)
    assert len(np.unique(pool_ids)) == len(pool_ids)
    assert not np.intersect1d(pool_ids, test_ids).size
    cv_base = dict(NETWORK_CANDIDATES["compact_regularized"])
    cv_grid = {}
    for gamma, lr, dropout, decay in product(
            (0.0, *CV_FOCAL_GAMMAS), CV_LEARNING_RATES, CV_DROPOUTS, CV_WEIGHT_DECAYS):
        family = "BCE" if gamma == 0 else "Focal"
        key = f"{family}_g{gamma:g}_lr{lr:g}_d{dropout:g}_wd{decay:g}"
        cv_grid[key] = dict(cv_base, focal_gamma=gamma, learning_rate=lr,
                            dropout=dropout, weight_decay=decay)
    cv_config = dict(outer_folds=CV_OUTER_FOLDS, inner_folds=CV_INNER_FOLDS,
        inner_seed=CV_INNER_SEED, outer_seeds=SEEDS, split_seed=SPLIT_SEED,
        stopping_fraction=CV_STOP_FRACTION, grid=cv_grid, selection_metric="mean inner AP")
    save_json(cv_config, RUN_DIR / "configs" / "nested_loss_cv.json")

### Fold-specific preprocessing

In [16]:
def cv_arrays(fit_candidates, score_indices, split_seed, *, pool_ids, pool_labels):
    fit_indices, stopping_indices = train_test_split(fit_candidates,
        test_size=CV_STOP_FRACTION, stratify=pool_labels[fit_candidates],
        random_state=split_seed)
    fold_preprocessor = clone(preprocessor)
    fold_preprocessor.fit(cohort.loc[pool_ids[fit_indices], FEATURES])
    arrays = tuple(fold_preprocessor.transform(cohort.loc[pool_ids[indices], FEATURES]).astype(np.float32)
                   for indices in (fit_indices, stopping_indices, score_indices))
    return arrays, fit_indices, stopping_indices, fold_preprocessor

### Inner search and outer evaluation

In [17]:
if RUN_NESTED_LOSS_CV:
    outer_cv = StratifiedKFold(n_splits=CV_OUTER_FOLDS, shuffle=True, random_state=SPLIT_SEED)
    inner_rows, outer_rows, oof_rows, partition_rows = [], [], [], []
    for outer, (development_indices, evaluation_indices) in enumerate(
            outer_cv.split(pool_ids, pool_labels), start=1):
        print(f"Outer fold {outer}/{CV_OUTER_FOLDS}: tuning {len(cv_grid)} configurations", flush=True)
        inner_cv = StratifiedKFold(n_splits=CV_INNER_FOLDS, shuffle=True, random_state=SPLIT_SEED + outer)
        fold_search = []
        for inner, (relative_fit, relative_score) in enumerate(
                inner_cv.split(development_indices, pool_labels[development_indices]), start=1):
            score_indices = development_indices[relative_score]
            arrays, fit_indices, stopping_indices, _ = cv_arrays(
                development_indices[relative_fit], score_indices, SPLIT_SEED + 100 * outer + inner,
                pool_ids=pool_ids, pool_labels=pool_labels)
            x_fit, x_stop, x_score = arrays
            for role, indices in (("fit", fit_indices), ("early_stop", stopping_indices), ("score", score_indices)):
                partition_rows.extend(dict(outer=outer, inner=inner, role=role, row_id=int(row_id))
                                      for row_id in pool_ids[indices])
            for candidate, hp in cv_grid.items():
                net, _, epoch = fit_model(x_fit, pool_labels[fit_indices], x_stop,
                    pool_labels[stopping_indices], weighted=False, seed=CV_INNER_SEED,
                    tag=candidate, hyperparameters=hp, save_artifacts=False, verbose=False)
                scores = expit(predict_logits(net, x_score))
                fold_search.append(dict(outer=outer, inner=inner, candidate=candidate,
                    loss_family=candidate.split("_", 1)[0], best_epoch=epoch,
                    ap=average_precision_score(pool_labels[score_indices], scores)))
            print(f"  Inner fold {inner}/{CV_INNER_FOLDS} complete", flush=True)
        inner_rows.extend(fold_search)
        ranking = (pd.DataFrame(fold_search).groupby(["loss_family", "candidate"], as_index=False)
                   .agg(mean_inner_ap=("ap", "mean"), sd_inner_ap=("ap", "std"))
                   .sort_values(["mean_inner_ap", "sd_inner_ap", "candidate"], ascending=[False, True, True]))
        winners = ranking.drop_duplicates("loss_family")
        arrays, fit_indices, stopping_indices, fold_preprocessor = cv_arrays(
            development_indices, evaluation_indices, SPLIT_SEED + 100 * outer,
            pool_ids=pool_ids, pool_labels=pool_labels)
        x_fit, x_stop, x_score = arrays
        for role, indices in (("fit", fit_indices), ("early_stop", stopping_indices), ("score", evaluation_indices)):
            partition_rows.extend(dict(outer=outer, inner=0, role=role, row_id=int(row_id))
                                  for row_id in pool_ids[indices])
        joblib.dump(fold_preprocessor, RUN_DIR / "checkpoints" / f"cv_outer{outer}_preprocessor.joblib")
        fold_features = fold_preprocessor.get_feature_names_out().tolist()
        labels = pool_labels[evaluation_indices]
        for winner in winners.itertuples(index=False):
            hp = cv_grid[winner.candidate]
            seed_predictions = []
            for seed in SEEDS:
                tag = f"cv_outer{outer}_{winner.loss_family}_seed{seed}"
                net, _, epoch = fit_model(x_fit, pool_labels[fit_indices], x_stop,
                    pool_labels[stopping_indices], weighted=False, seed=seed,
                    tag=tag, hyperparameters=hp, feature_names=fold_features, verbose=False)
                seed_predictions.append(expit(predict_logits(net, x_score)))
            scores = np.mean(seed_predictions, axis=0)
            outer_rows.append(dict(outer=outer, loss_family=winner.loss_family,
                candidate=winner.candidate, mean_inner_ap=winner.mean_inner_ap,
                n=len(labels), positives=int(labels.sum()), ap=average_precision_score(labels, scores),
                auc=roc_auc_score(labels, scores), brier=brier_score_loss(labels, scores)))
            oof_rows.extend(dict(outer=outer, loss_family=winner.loss_family,
                row_id=int(row_id), label=int(label), score=float(score))
                for row_id, label, score in zip(pool_ids[evaluation_indices], labels, scores))
            print(f"  {winner.loss_family}: {winner.candidate}, outer AP {outer_rows[-1]['ap']:.4f}", flush=True)
        pd.DataFrame(inner_rows).to_csv(RUN_DIR / "metrics" / "cv_inner_search.csv", index=False)
        pd.DataFrame(outer_rows).to_csv(RUN_DIR / "metrics" / "cv_outer_metrics.csv", index=False)


Outer fold 1/5: tuning 24 configurations
  Inner fold 1/3 complete
  Inner fold 2/3 complete
  Inner fold 3/3 complete
  BCE: BCE_g0_lr0.0005_d0.2_wd0.001, outer AP 0.4313
  Focal: Focal_g1_lr0.0005_d0.2_wd0.001, outer AP 0.4394
Outer fold 2/5: tuning 24 configurations
  Inner fold 1/3 complete
  Inner fold 2/3 complete
  Inner fold 3/3 complete
  BCE: BCE_g0_lr0.0005_d0.2_wd0.0001, outer AP 0.3956
  Focal: Focal_g1_lr0.001_d0.1_wd0.0001, outer AP 0.3904
Outer fold 3/5: tuning 24 configurations
  Inner fold 1/3 complete
  Inner fold 2/3 complete
  Inner fold 3/3 complete
  BCE: BCE_g0_lr0.001_d0.1_wd0.0001, outer AP 0.3260
  Focal: Focal_g1_lr0.001_d0.2_wd0.001, outer AP 0.3330
Outer fold 4/5: tuning 24 configurations
  Inner fold 1/3 complete
  Inner fold 2/3 complete
  Inner fold 3/3 complete
  BCE: BCE_g0_lr0.001_d0.1_wd0.001, outer AP 0.3126
  Focal: Focal_g1_lr0.0005_d0.1_wd0.0001, outer AP 0.3145
Outer fold 5/5: tuning 24 configurations
  Inner fold 1/3 complete
  Inner fold 2/3 

### Paired results and out-of-fold predictions

In [18]:
if RUN_NESTED_LOSS_CV:
    outer_results = pd.DataFrame(outer_rows)
    oof = pd.DataFrame(oof_rows)
    assert not oof.duplicated(["loss_family", "row_id"]).any()
    assert all(set(group.row_id) == set(pool_ids) for _, group in oof.groupby("loss_family"))
    oof.to_csv(RUN_DIR / "predictions" / "cv_oof_predictions.csv", index=False)
    pd.DataFrame(partition_rows).to_csv(RUN_DIR / "configs" / "cv_partitions.csv", index=False)
    metrics = ["ap", "auc", "brier"]
    cv_summary = outer_results.groupby("loss_family")[metrics].agg(["mean", "std"])
    paired_differences = pd.DataFrame({
        metric: (wide["Focal"] - wide["BCE"])
        for metric in metrics
        for wide in [outer_results.pivot(index="outer", columns="loss_family", values=metric)]
    })
    cv_summary.to_csv(RUN_DIR / "metrics" / "cv_summary.csv")
    paired_differences.to_csv(RUN_DIR / "metrics" / "cv_focal_minus_bce.csv")
    display(cv_summary.round(4))
    display(paired_differences.round(4))
    print(f"Mean paired focal-minus-BCE AP: {paired_differences.ap.mean():+.4f}")


ap             auc           brier        
               mean     std    mean     std    mean     std
loss_family                                                
BCE          0.3716  0.0504  0.7424  0.0195  0.1046  0.0039
Focal        0.3746  0.0506  0.7423  0.0209  0.1170  0.0026

,ap,auc,brier
outer,,,
1,0.0081,0.0001,0.0146
2,-0.0052,0.0015,0.0142
3,0.0070,-0.0055,0.0106
4,0.0019,0.0004,0.0114
5,0.0031,0.0031,0.0115


Mean paired focal-minus-BCE AP: +0.0030


## 9. Locked test evaluation and calibration
All models, calibrators and thresholds are now frozen. Test results are descriptive final assessments, not another selection loop. Report both 0.5 and validation-selected thresholds for raw and optional Platt-calibrated probabilities. AP denotes average precision, not trapezoidal PR area. ECE uses ten fixed-width probability bins; it is sample-size/binning dependent. Platt calibration is exploratory because the disjoint calibration set is small. No calibration method is chosen from test results.


In [19]:
if RUN_FINAL_EVALUATION:
    def safe_divide(a, b):
        return float(a / b) if b else np.nan

    def ece(labels, probability, bins=10):
        indices = np.minimum((np.asarray(probability) * bins).astype(int), bins - 1)
        return sum(np.mean(indices == k) * abs(np.mean(np.asarray(labels)[indices == k]) - np.mean(probability[indices == k]))
                   for k in range(bins) if np.any(indices == k))

    def evaluate(labels, probability, threshold):
        labels, probability = np.asarray(labels), np.asarray(probability)
        tn, fp, fn, tp = confusion_matrix(labels, probability >= threshold, labels=[0, 1]).ravel()
        both = len(np.unique(labels)) == 2
        return dict(roc_auc=roc_auc_score(labels, probability) if both else np.nan,
            average_precision=average_precision_score(labels, probability) if both else np.nan,
            sensitivity=safe_divide(tp, tp + fn), specificity=safe_divide(tn, tn + fp),
            fpr=safe_divide(fp, tn + fp), precision=safe_divide(tp, tp + fp),
            f1=safe_divide(2 * tp, 2 * tp + fp + fn),
            balanced_accuracy=(safe_divide(tp, tp + fn) + safe_divide(tn, tn + fp)) / 2,
            brier=brier_score_loss(labels, probability), ece=ece(labels, probability),
            tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

    test_array = transform(cohort.loc[test_ids])
    test_y = y_all.loc[test_ids].to_numpy(dtype=int)
    prediction_frames, metric_rows = [], []
    for tag, run in experiments.items():
        logits = predict_logits(run["model"], test_array)
        probabilities = {"raw": (expit(logits), run["threshold"])}
        if run["calibrator"] is not None:
            probabilities["platt"] = (run["calibrator"].predict_proba(logits.reshape(-1, 1))[:, 1], run["calibrated_threshold"])
        for kind, (p, selected_threshold) in probabilities.items():
            frame = pd.DataFrame(dict(row_id=test_ids, y_true=test_y, probability=p,
                logit=logits, model=run["name"], seed=run["seed"], calibration=kind,
                selected_threshold=selected_threshold))
            frame.to_csv(RUN_DIR / "predictions" / f"{tag}_{kind}.csv", index=False)
            prediction_frames.append(frame)
            for policy, threshold in [("fixed_0.5", 0.5), ("validation_balanced_accuracy", selected_threshold)]:
                metric_rows.append(dict(model=run["name"], seed=run["seed"], calibration=kind,
                    policy=policy, threshold=threshold, **evaluate(test_y, p, threshold)))
    predictions = pd.concat(prediction_frames, ignore_index=True)
    metrics = pd.DataFrame(metric_rows)
    metrics.to_csv(RUN_DIR / "metrics" / "test_metrics.csv", index=False)
    SCORES = ["roc_auc", "average_precision", "sensitivity", "specificity", "precision", "f1", "balanced_accuracy", "brier", "ece"]
    summary = metrics.groupby(["model", "calibration", "policy"])[SCORES].agg(["mean", "std"])
    summary.to_csv(RUN_DIR / "metrics" / "seed_summary.csv")
    display(summary.round(3))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.7), layout="constrained")
    for name, color in [("MLP-A", "#27678b"), ("MLP-B", "#cf7043")]:
        for kind, style in [("raw", "-"), ("platt", "--")]:
            selected = predictions[(predictions.model == name) & (predictions.seed == SEEDS[0]) & (predictions.calibration == kind)]
            if selected.empty:
                continue
            p = selected.probability.to_numpy()
            label = f"{name} {kind}; seed {SEEDS[0]}"
            fpr, tpr, _ = roc_curve(test_y, p)
            precision, recall, _ = precision_recall_curve(test_y, p)
            fraction, mean_probability = calibration_curve(test_y, p, n_bins=8, strategy="quantile")
            axes[0].plot(fpr, tpr, style, color=color, label=label)
            axes[1].plot(recall, precision, style, color=color, label=label)
            axes[2].plot(mean_probability, fraction, style, color=color, marker="o", label=label)
    axes[0].plot([0, 1], [0, 1], ":", color="grey")
    axes[1].axhline(test_y.mean(), linestyle=":", color="grey", label="Test prevalence")
    axes[2].plot([0, 1], [0, 1], ":", color="grey")
    for ax, title, xlabel, ylabel in zip(axes, ["ROC", "Precision-recall", "Reliability (quantile bins)"],
            ["False positive rate", "Recall", "Mean predicted probability"],
            ["Sensitivity", "Precision", "Observed event fraction"]):
        ax.set(title=title, xlabel=xlabel, ylabel=ylabel, xlim=(0, 1), ylim=(0, 1))
        ax.legend(fontsize=7)
    save_figure("roc_pr_calibration_first_prespecified_seed")
    fig, axes = plt.subplots(2, 2, figsize=(9, 8), layout="constrained")
    for row, name in enumerate(["MLP-A", "MLP-B"]):
        r = experiments[f"{name}_seed{SEEDS[0]}"]
        p = expit(predict_logits(r["model"], test_array))
        for col, (policy, threshold) in enumerate([("0.5", 0.5), ("validation", r["threshold"]) ]):
            matrix = confusion_matrix(test_y, p >= threshold, labels=[0, 1])
            axes[row, col].imshow(matrix, cmap="Blues")
            for i in range(2):
                for j in range(2):
                    axes[row, col].text(j, i, str(matrix[i, j]), ha="center", va="center",
                        color="white" if matrix[i, j] > matrix.max() / 2 else "black")
            axes[row, col].set(title=f"{name}, {policy}, seed {SEEDS[0]}", xlabel="Predicted class", ylabel="Actual class", xticks=[0, 1], yticks=[0, 1])
    save_figure("confusion_matrices")
else:
    print("Skipped: RUN_FINAL_EVALUATION=False; network development uses training/validation only.")


Skipped: RUN_FINAL_EVALUATION=False; network development uses training/validation only.


## 10. Fairness with uncertainty
Evaluate Race, Age and Age × Race for each seed, calibration variant and threshold policy. TPR/FPR use their respective positive/negative denominators; undefined rates remain NaN. Wilson 95% intervals cover TPR, specificity and FPR. Brier/ECE intervals use patient bootstrap; ECE is omitted unless n≥50 and both classes≥10. Groups below n=30 or either class<10 are flagged as underpowered, and calibration bins require the larger threshold. Disparities are descriptive, without automatic demographic-parity targets or causal claims. Bootstrap intervals are pointwise and do not adjust for multiple comparisons. Seed SD is not a patient-sampling confidence interval.


In [20]:
if RUN_FINAL_EVALUATION:
    demographics = cohort.loc[test_ids, ["row_id", "Age", "Race"]].copy()
    demographics["age_group"] = pd.cut(demographics.Age, bins=AGE_BINS, labels=AGE_LABELS, right=False).astype(str)
    demographics["intersection"] = demographics.age_group + " | " + demographics.Race
    demographics = demographics.reset_index(drop=True)

    def wilson(successes, total):
        if not total:
            return np.nan, np.nan
        z = 1.959964
        p = successes / total
        center = (p + z*z/(2*total)) / (1 + z*z/total)
        half = z * np.sqrt(p*(1-p)/total + z*z/(4*total*total)) / (1+z*z/total)
        return center-half, center+half

    def bootstrap_metric(labels, probability, threshold, metric, seed=SPLIT_SEED):
        rng = np.random.default_rng(seed)
        values = []
        for _ in range(BOOTSTRAPS):
            take = rng.integers(0, len(labels), len(labels))
            if metric == "brier":
                value = np.mean((labels[take] - probability[take]) ** 2)
            else:
                value = evaluate(labels[take], probability[take], threshold)[metric]
            if np.isfinite(value):
                values.append(value)
        return tuple(np.quantile(values, [0.025, 0.975])) if values else (np.nan, np.nan)

    def fairness_for_frame(frame, policy, threshold):
        joined = frame.merge(demographics, on="row_id", validate="one_to_one")
        rows = []
        for dimension in ["Race", "age_group", "intersection"]:
            for group, chunk in joined.groupby(dimension, observed=True):
                yy, pp = chunk.y_true.to_numpy(), chunk.probability.to_numpy()
                m = evaluate(yy, pp, threshold)
                n, positives = len(yy), int(yy.sum())
                negatives = n - positives
                sufficient = n >= MIN_GROUP_N and min(positives, negatives) >= MIN_GROUP_CLASS
                calibration_ok = n >= 50 and min(positives, negatives) >= 10
                row = dict(dimension=dimension, group=group, n=n, positives=positives,
                    negatives=negatives, underpowered=not sufficient, calibration_eligible=calibration_ok,
                    policy=policy, threshold=threshold, **m)
                for metric, successes, total in [("sensitivity", m["tp"], positives),
                        ("specificity", m["tn"], negatives), ("fpr", m["fp"], negatives)]:
                    row[f"{metric}_low"], row[f"{metric}_high"] = wilson(successes, total)
                row["brier_low"], row["brier_high"] = bootstrap_metric(yy, pp, threshold, "brier")
                if calibration_ok:
                    row["ece_low"], row["ece_high"] = bootstrap_metric(yy, pp, threshold, "ece")
                else:
                    row["ece"] = row["ece_low"] = row["ece_high"] = np.nan
                rows.append(row)
        return pd.DataFrame(rows)

    fairness_tables = []
    for (name, seed, kind), frame in predictions.groupby(["model", "seed", "calibration"]):
        for policy, threshold in [("fixed_0.5", 0.5), ("validation_balanced_accuracy", frame.selected_threshold.iloc[0])]:
            table = fairness_for_frame(frame, policy, threshold)
            fairness_tables.append(table.assign(model=name, seed=seed, calibration=kind))
    fairness = pd.concat(fairness_tables, ignore_index=True)
    fairness.to_csv(RUN_DIR / "fairness" / "subgroup_metrics_intervals.csv", index=False)

    def disparity_table(table):
        output = []
        keys = ["model", "seed", "calibration", "policy", "dimension"]
        for key, chunk in table.groupby(keys):
            for scope, subset in [("all_estimable", chunk), ("adequate_size_only", chunk.loc[~chunk.underpowered])]:
                tpr, fpr = subset.sensitivity.dropna(), subset.fpr.dropna()
                tgap = float(tpr.max() - tpr.min()) if len(tpr) >= 2 else np.nan
                fgap = float(fpr.max() - fpr.min()) if len(fpr) >= 2 else np.nan
                output.append(dict(zip(keys, key), scope=scope, groups=len(subset),
                    equal_opportunity_difference=tgap, fpr_difference=fgap,
                    equalized_odds_max_gap=max(tgap, fgap) if np.isfinite([tgap, fgap]).all() else np.nan))
        return pd.DataFrame(output)

    disparities = disparity_table(fairness)
    disparities.to_csv(RUN_DIR / "fairness" / "disparities.csv", index=False)
    display(disparities[(disparities.seed == SEEDS[0]) & (disparities.calibration == "raw")].round(3))
    # Reliability curves for sufficiently populated Race/Age groups, first prespecified seed.
    fig, axes = plt.subplots(2, 2, figsize=(12, 9), layout="constrained")
    for row, name in enumerate(["MLP-A", "MLP-B"]):
        frame = predictions[(predictions.model == name) & (predictions.seed == SEEDS[0]) & (predictions.calibration == "raw")]
        joined = frame.merge(demographics, on="row_id", validate="one_to_one")
        for col, dimension in enumerate(["Race", "age_group"]):
            ax = axes[row, col]
            for group, chunk in joined.groupby(dimension):
                if len(chunk) < 50 or min(chunk.y_true.sum(), len(chunk)-chunk.y_true.sum()) < 10:
                    continue
                observed_rate, predicted_rate = calibration_curve(chunk.y_true, chunk.probability, n_bins=4, strategy="quantile")
                ax.plot(predicted_rate, observed_rate, marker="o", label=f"{group} (n={len(chunk)})")
            ax.plot([0, 1], [0, 1], ":", color="grey")
            ax.set(title=f"{name}: {dimension}, seed {SEEDS[0]}", xlabel="Predicted probability", ylabel="Observed event fraction", xlim=(0, 1), ylim=(0, 1))
            if ax.get_legend_handles_labels()[0]:
                ax.legend(fontsize=8)
    save_figure("subgroup_calibration")
else:
    print("Skipped: RUN_FINAL_EVALUATION=False; network development uses training/validation only.")


Skipped: RUN_FINAL_EVALUATION=False; network development uses training/validation only.


## 11. Optional paired baseline imports
Expected prediction CSV: `row_id,y_true,probability,model`; one row per test patient per model, probabilities for class 1. Every model must cover **exactly the same test IDs and labels**, with finite probabilities in [0,1]. Mismatches stop rather than silently intersect cohorts. Baseline comparison uses raw probabilities at 0.5 (no test-optimized threshold). Paired bootstrap AP/ROC-AUC/Brier differences use the same sampled patient IDs for each pair and compare each MLP seed separately. Baseline fairness uses the identical subgroup function. No baseline is retrained or invented.

In the existing notebook export `row_id` through the original CSV→cleaned→task mapping, then export `model.predict_proba(X_test)[:,1]` and aligned `y_test`. A task DataFrame index from `pd.concat(...,ignore_index=True)` is not the original CSV index. The mapping is reproduced here by `cohort` order and `clean_id`.


In [21]:
if RUN_FINAL_EVALUATION:
    baseline_metrics, paired_rows = [], []
    baseline_fairness = pd.DataFrame()
    if BASELINE_PATH:
        baseline = pd.read_csv(BASELINE_PATH)
        assert {"row_id", "y_true", "probability", "model"}.issubset(baseline.columns)
        assert baseline.model.notna().all()
        baseline_fairness_tables = []
        for name, frame in baseline.groupby("model"):
            assert not frame.row_id.duplicated().any()
            assert set(frame.row_id) == set(test_ids), f"Test ID mismatch for {name}"
            frame = frame.set_index("row_id").loc[test_ids].reset_index()
            assert np.array_equal(frame.y_true.to_numpy(), test_y)
            assert np.isfinite(frame.probability).all() and frame.probability.between(0, 1).all()
            bp = frame.probability.to_numpy()
            baseline_metrics.append(dict(model=name, seed="baseline", **evaluate(test_y, bp, 0.5)))
            baseline_fairness_tables.append(fairness_for_frame(frame, "fixed_0.5", 0.5).assign(model=name, seed="baseline", calibration="raw"))
            for tag, run in experiments.items():
                fp = predictions[(predictions.model == run["name"]) & (predictions.seed == run["seed"]) & (predictions.calibration == "raw")].set_index("row_id").loc[test_ids]
                mp = fp.probability.to_numpy()
                rng = np.random.default_rng(SPLIT_SEED)
                for metric in ["roc_auc", "average_precision", "brier"]:
                    differences = []
                    for _ in range(BOOTSTRAPS):
                        take = rng.integers(0, len(test_y), len(test_y))
                        a, b = evaluate(test_y[take], mp[take], 0.5), evaluate(test_y[take], bp[take], 0.5)
                        if np.isfinite(a[metric]) and np.isfinite(b[metric]):
                            differences.append(a[metric] - b[metric])
                    low, high = np.quantile(differences, [0.025, 0.975])
                    paired_rows.append(dict(baseline=name, model=tag, metric=metric,
                        difference=evaluate(test_y, mp, 0.5)[metric]-evaluate(test_y, bp, 0.5)[metric], low=low, high=high))
        baseline_fairness = pd.concat(baseline_fairness_tables, ignore_index=True)
        baseline_fairness.to_csv(RUN_DIR / "fairness" / "baseline_subgroups.csv", index=False)
        disparity_table(baseline_fairness).to_csv(RUN_DIR / "fairness" / "baseline_disparities.csv", index=False)
        pd.DataFrame(paired_rows).to_csv(RUN_DIR / "metrics" / "paired_baseline_differences.csv", index=False)
    else:
        print("Baseline predictions unavailable: no baseline results or paired comparisons fabricated.")
    comparison = pd.concat([pd.DataFrame(baseline_metrics),
        metrics[(metrics.calibration == "raw") & (metrics.policy == "fixed_0.5")]], ignore_index=True)
    comparison.to_csv(RUN_DIR / "metrics" / "baseline_comparison.csv", index=False)
    display(comparison[["model", "seed", "roc_auc", "average_precision", "sensitivity", "specificity", "brier"]].round(3))
else:
    print("Skipped: RUN_FINAL_EVALUATION=False; network development uses training/validation only.")


Skipped: RUN_FINAL_EVALUATION=False; network development uses training/validation only.


## 12. Bounded measurement-sensitivity scenarios
Perturb tumor diameter by ±1 mm **only where the resulting size remains in the same T-stage diameter band** (T1≤20, T2>20–50, T3>50; T4 is not size-defined), while preserving age, all node counts, receptors and staging. Rows with original T-size contradictions, nonpositive size or node anomalies are left unchanged and counted. This is a bounded illustrative measurement scenario, not a verified error distribution or proof of clinical robustness. T stage can reflect more than diameter; seek domain review before expanding scenarios. We do not independently perturb node counts or stage categories. Report mean/95th-percentile |Δp|, signed Δp and decision flips at both thresholds, per seed and calibration variant.


In [22]:
if RUN_FINAL_EVALUATION:
    def size_stage_consistent(frame):
        size, stage = frame["Tumor Size"], frame["T Stage"]
        return ((stage.eq("T1") & size.gt(0) & size.le(20)) |
                (stage.eq("T2") & size.gt(20) & size.le(50)) |
                (stage.eq("T3") & size.gt(50)) |
                (stage.eq("T4") & size.gt(0)))

    robustness_rows = []
    original = cohort.loc[test_ids, FEATURES].copy()
    for delta in [-1.0, 1.0]:
        perturbed = original.copy()
        perturbed["Tumor Size"] = perturbed["Tumor Size"] + delta
        eligible = (size_stage_consistent(original) & size_stage_consistent(perturbed) &
            original["Reginol Node Positive"].le(original["Regional Node Examined"]) &
            original[NUM_COLS].ge(0).all(axis=1))
        perturbed.loc[~eligible, "Tumor Size"] = original.loc[~eligible, "Tumor Size"]
        assert (perturbed.loc[eligible, "Reginol Node Positive"] <= perturbed.loc[eligible, "Regional Node Examined"]).all()
        changed_array = transform(perturbed)
        for tag, run in experiments.items():
            logits = predict_logits(run["model"], changed_array)
            for kind in (["raw", "platt"] if run["calibrator"] is not None else ["raw"]):
                p = expit(logits) if kind == "raw" else run["calibrator"].predict_proba(logits.reshape(-1, 1))[:, 1]
                base = predictions[(predictions.model == run["name"]) & (predictions.seed == run["seed"]) & (predictions.calibration == kind)].set_index("row_id").loc[test_ids].probability.to_numpy()
                mask = eligible.to_numpy()
                differences = p[mask] - base[mask]
                selected = run["threshold"] if kind == "raw" else run["calibrated_threshold"]
                for policy, threshold in [("fixed_0.5", 0.5), ("validation_balanced_accuracy", selected)]:
                    robustness_rows.append(dict(model=run["name"], seed=run["seed"], calibration=kind,
                        delta_mm=delta, policy=policy, threshold=threshold, changed_n=int(mask.sum()),
                        unchanged_n=int((~mask).sum()), mean_absolute_delta=float(np.abs(differences).mean()) if len(differences) else np.nan,
                        p95_absolute_delta=float(np.quantile(np.abs(differences), 0.95)) if len(differences) else np.nan,
                        mean_signed_delta=float(differences.mean()) if len(differences) else np.nan,
                        flip_rate_changed=float(np.mean((p[mask] >= threshold) != (base[mask] >= threshold))) if len(differences) else np.nan,
                        test_ap_delta=evaluate(test_y, p, threshold)["average_precision"]-evaluate(test_y, base, threshold)["average_precision"]))
    robustness = pd.DataFrame(robustness_rows)
    robustness.to_csv(RUN_DIR / "metrics" / "robustness.csv", index=False)
    display(robustness.round(4))
else:
    print("Skipped: RUN_FINAL_EVALUATION=False; network development uses training/validation only.")


Skipped: RUN_FINAL_EVALUATION=False; network development uses training/validation only.


## 13. SHAP global, local and seed stability
Use one fixed background from original training patients and one fixed real test subset across all six networks, independent of performance. Explain **raw class-1 logits** (positive values raise death-risk log-odds), not calibrated probabilities. Include the first prespecified seed's TP/TN/FP/FN when available for both networks; never choose a best test seed. Expected gradients are approximate: save additivity residuals and the Monte Carlo sample count. One-hot contributions are summed per patient back to each original feature **before** computing mean absolute grouped importance. An encoded-category beeswarm and a grouped bar plot answer different questions.

Correlated Tumor Size/T Stage/N Stage/6th Stage/node counts can redistribute attributions; expected-gradient interpolation may traverse unrealistic one-hot/category combinations. The explanations are model associations, not causal or clinical effects. Rank correlation/top-five overlap across seeds do not validate correctness. Optional baseline attribution CSV schema: `row_id,model,feature,attribution`, one row per explained test patient × original feature, on the same logit scale, using the same training background and a compatible attribution method. Only import after confirming that provenance explicitly.


In [23]:
if RUN_FINAL_EVALUATION:
    BASELINE_SHAP_COMPATIBLE = False  # Set True only after verifying scale/background/method.
    shap_rankings, shap_stability_rows, grouped_attributions = [], [], {}
    if RUN_SHAP:
        import shap
        rng = np.random.default_rng(SPLIT_SEED)
        background_positions = rng.choice(len(train_array), min(SHAP_BACKGROUND, len(train_array)), replace=False)
        explain_positions = set(rng.choice(len(test_array), min(SHAP_PATIENTS, len(test_array)), replace=False).tolist())
        representatives = {}
        for name in ["MLP-A", "MLP-B"]:
            run = experiments[f"{name}_seed{SEEDS[0]}"]
            p = expit(predict_logits(run["model"], test_array))
            predicted = p >= run["threshold"]
            for label, truth, decision in [("TP", 1, 1), ("TN", 0, 0), ("FP", 0, 1), ("FN", 1, 0)]:
                candidates = np.flatnonzero((test_y == truth) & (predicted == decision))
                if len(candidates):
                    position = int(candidates[0])
                    explain_positions.add(position)
                    representatives[(name, label)] = position
        explain_positions = np.array(sorted(explain_positions), dtype=int)
        background = torch.from_numpy(train_array[background_positions]).to(device)
        explained = torch.from_numpy(test_array[explain_positions]).to(device)
        np.save(RUN_DIR / "explainability" / "training_background_row_ids.npy", np.asarray(train_ids)[background_positions])
        np.save(RUN_DIR / "explainability" / "explained_test_row_ids.npy", np.asarray(test_ids)[explain_positions])
        for tag, run in experiments.items():
            model = run["model"].eval()
            seed_everything(SPLIT_SEED)  # Common Monte Carlo random numbers.
            explainer = shap.GradientExplainer(model, background, batch_size=32)
            values = explainer.shap_values(explained, nsamples=SHAP_SAMPLES, rseed=SPLIT_SEED)
            if isinstance(values, list):
                values = values[0]
            values = np.asarray(values)
            if values.ndim == 3 and values.shape[-1] == 1:
                values = values[..., 0]
            assert values.shape == (len(explain_positions), INPUT_DIM)
            np.savez_compressed(RUN_DIR / "explainability" / f"{tag}_shap.npz",
                values=values, row_ids=np.asarray(test_ids)[explain_positions], encoded_features=np.asarray(encoded_names))
            with torch.no_grad():
                expected_logit = model(background).mean().item()
            actual_logit = predict_logits(model, test_array[explain_positions])
            pd.DataFrame(dict(row_id=np.asarray(test_ids)[explain_positions], expected_logit=expected_logit,
                actual_logit=actual_logit, attributed_sum=values.sum(axis=1),
                additivity_residual=actual_logit-expected_logit-values.sum(axis=1))).to_csv(
                    RUN_DIR / "explainability" / f"{tag}_additivity.csv", index=False)
            grouped = pd.DataFrame({feature: values[:, np.asarray(encoded_groups) == feature].sum(axis=1) for feature in FEATURES},
                index=np.asarray(test_ids)[explain_positions])
            grouped.index.name = "row_id"
            grouped.to_csv(RUN_DIR / "explainability" / f"{tag}_grouped_attributions.csv")
            grouped_attributions[tag] = grouped
            importance = grouped.abs().mean().sort_values(ascending=False)
            shap_rankings.extend(dict(model=run["name"], seed=run["seed"], feature=f, mean_absolute_logit_shap=float(v)) for f, v in importance.items())
            if run["seed"] == SEEDS[0]:
                shap.summary_plot(values, test_array[explain_positions], feature_names=encoded_names, show=False, max_display=20)
                plt.title(f"{tag}: encoded-category logit attributions")
                save_figure(f"shap_summary_{tag}")
                plt.figure(figsize=(8, 5))
                importance.sort_values().plot.barh(color="#27678b")
                plt.title(f"{tag}: original-feature importance")
                plt.xlabel("Mean absolute summed logit attribution")
                save_figure(f"shap_grouped_{tag}")
                for (name, category), position in representatives.items():
                    if name != run["name"]:
                        continue
                    row_id = int(test_ids[position])
                    signed = grouped.loc[row_id].sort_values()
                    plt.figure(figsize=(8, 5))
                    signed.plot.barh(color=["#cf7043" if v > 0 else "#27678b" for v in signed])
                    plt.axvline(0, color="grey", linewidth=1)
                    plt.xlabel("Contribution to class-1 logit")
                    plt.title(f"{tag} {category}; row {row_id}; base logit={expected_logit:.2f}")
                    save_figure(f"shap_local_{tag}_{category}_row{row_id}")
        ranking_table = pd.DataFrame(shap_rankings)
        ranking_table.to_csv(RUN_DIR / "explainability" / "grouped_importance.csv", index=False)
        tags = list(grouped_attributions)
        for i, first in enumerate(tags):
            for second in tags[i+1:]:
                a, b = grouped_attributions[first].abs().mean(), grouped_attributions[second].abs().mean()
                shap_stability_rows.append(dict(first=first, second=second,
                    rank_spearman=float(spearmanr(a, b).statistic),
                    top5_overlap=len(set(a.nlargest(5).index) & set(b.nlargest(5).index))/5,
                    mean_absolute_attribution_difference=float(np.abs(grouped_attributions[first]-grouped_attributions[second]).to_numpy().mean())))
        if BASELINE_SHAP_PATH:
            assert BASELINE_SHAP_COMPATIBLE, "Confirm common scale/background/method before importing explanations."
            baseline_shap = pd.read_csv(BASELINE_SHAP_PATH)
            assert {"row_id", "model", "feature", "attribution"}.issubset(baseline_shap.columns)
            for name, frame in baseline_shap.groupby("model"):
                assert not frame.duplicated(["row_id", "feature"]).any()
                assert set(frame.row_id) == set(np.asarray(test_ids)[explain_positions])
                assert set(frame.feature) == set(FEATURES)
                wide = frame.pivot(index="row_id", columns="feature", values="attribution").reindex(index=np.asarray(test_ids)[explain_positions], columns=FEATURES)
                assert np.isfinite(wide.to_numpy()).all()
                for tag, grouped in grouped_attributions.items():
                    a, b = grouped.abs().mean(), wide.abs().mean()
                    shap_stability_rows.append(dict(first=tag, second=str(name),
                        rank_spearman=float(spearmanr(a, b).statistic),
                        top5_overlap=len(set(a.nlargest(5).index) & set(b.nlargest(5).index))/5,
                        mean_absolute_attribution_difference=float(np.abs(grouped-wide).to_numpy().mean())))
        pd.DataFrame(shap_stability_rows).to_csv(RUN_DIR / "explainability" / "stability.csv", index=False)
        display(ranking_table.groupby(["model", "feature"]).mean_absolute_logit_shap.agg(["mean", "std"]).round(4))
    else:
        print("SHAP disabled by configuration; report will mark explanation analysis incomplete.")
else:
    print("Skipped: RUN_FINAL_EVALUATION=False; network development uses training/validation only.")


Skipped: RUN_FINAL_EVALUATION=False; network development uses training/validation only.


## 14. Automated research report and future augmentation readiness
The report below is assembled only from this run. It does not certify fairness or clinical suitability. Read subgroup confidence intervals, denominator flags, calibration and optimization variability together. A stable seed average alone is insufficient evidence to approve augmentation: require training-only generator provenance, plausible synthetic records, repeated-split/external validation and domain review. The same original preprocessor, fixed partitions, architecture and evaluation functions can then be reused.


In [24]:
if RUN_FINAL_EVALUATION:
    raw_selected = metrics[(metrics.calibration == "raw") & (metrics.policy == "validation_balanced_accuracy")]
    aggregate = raw_selected.groupby("model")[SCORES].agg(["mean", "std"])
    report = ["# SEER MLP benchmark — computed run report", f"Run: {RUN_DIR.name}",
        f"Network candidate selected by development validation: {selected_candidate}",
        f"Dataset SHA256: {AUDIT['dataset_sha256']}",
        f"Eligible records: {len(cohort)}; excluded early-censored: {len(excluded)}. Split: {split_origin}.",
        "## Predictive performance",
        "Raw probabilities; validation-balanced-accuracy thresholds; mean ± sample SD across fixed-split seeds."]
    for name in ["MLP-A", "MLP-B"]:
        row = aggregate.loc[name]
        report.append(name + ": " + "; ".join(f"{m}={row[(m, 'mean')]:.3f} ± {row[(m, 'std')]:.3f}" for m in SCORES))
    recall_delta = aggregate.loc["MLP-B", ("sensitivity", "mean")] - aggregate.loc["MLP-A", ("sensitivity", "mean")]
    report.append(f"Weighted-minus-standard sensitivity difference at validation-selected thresholds: {recall_delta:+.3f}. This is descriptive, not a significance test.")
    report.extend(["## Calibration", "Brier and ECE by prespecified probability variant (lower is better):"])
    for (name, kind), frame in metrics[metrics.policy == "fixed_0.5"].groupby(["model", "calibration"]):
        report.append(f"{name}/{kind}: Brier={frame.brier.mean():.3f}; ECE={frame.ece.mean():.3f}.")
    report.append("Platt parameters use disjoint training-derived validation records; the small calibration sample limits reliability. Test plots must not be used to choose a calibration variant.")
    report.extend(["## Fairness", "Raw, validation-threshold disparities, averaged across seeds; uncertainty and sample sizes are in fairness/subgroup_metrics_intervals.csv."])
    selected_disparities = disparities[(disparities.calibration == "raw") & (disparities.policy == "validation_balanced_accuracy") & (disparities.scope == "all_estimable")]
    for (name, dimension), frame in selected_disparities.groupby(["model", "dimension"]):
        report.append(f"{name}/{dimension}: mean TPR gap={frame.equal_opportunity_difference.mean():.3f}; FPR gap={frame.fpr_difference.mean():.3f}.")
    report.append(f"Underpowered subgroup rows: {int(fairness.underpowered.sum())}/{len(fairness)} (repeated across seed/policy/calibration). Pointwise intervals and sparse denominators preclude a blanket fairness conclusion.")
    tradeoff = selected_disparities.pivot_table(index="dimension", columns="model", values="equalized_odds_max_gap")
    for dimension, row in tradeoff.iterrows():
        report.append(f"{dimension}: weighted-minus-standard mean equalized-odds maximum-gap difference={row['MLP-B']-row['MLP-A']:+.3f}; compare with sensitivity difference {recall_delta:+.3f} above.")
    report.append("## Explainability")
    if shap_rankings:
        ranks = pd.DataFrame(shap_rankings)
        for name, frame in ranks.groupby("model"):
            top = frame.groupby("feature").mean_absolute_logit_shap.mean().nlargest(5)
            report.append(f"{name} leading original features: {', '.join(top.index)}.")
        stability = pd.DataFrame(shap_stability_rows)
        report.append(f"Pairwise attribution rank Spearman range: {stability.rank_spearman.min():.3f} to {stability.rank_spearman.max():.3f}. Includes seed/weight comparisons; see stability.csv for pairs.")
        report.append("Attributions explain raw logits, not causation. Correlated staging/nodal features and interpolation limit interpretation; inspect saved additivity residuals.")
    else:
        report.append("Not evaluated: SHAP was disabled. Benchmark explainability assessment is incomplete.")
    report.extend(["## Robustness", f"Across bounded ±1 mm stage-consistent size scenarios: largest mean |Δp|={robustness.mean_absolute_delta.max():.4f}; largest changed-record decision-flip fraction={robustness.flip_rate_changed.max():.3f}.",
        "The perturbations cover a narrow measurement scenario. Seed SD concerns optimization at one split and does not quantify cohort or external-site uncertainty.",
        "## Baseline comparison"])
    if baseline_metrics:
        for row in baseline_metrics:
            report.append(f"{row['model']}: AUC={row['roc_auc']:.3f}; AP={row['average_precision']:.3f}; Brier={row['brier']:.3f} at threshold 0.5 on exactly aligned patients.")
        report.append("Paired per-seed bootstrap differences and baseline fairness are saved. Negative Brier differences favor MLP; positive AP/AUC differences favor MLP. Pointwise intervals are exploratory.")
    else:
        report.append("No baseline predictions imported; comparative performance/fairness is unavailable.")
    report.extend(["## Suitability for synthetic-data experiments",
        "The code exposes a fixed-preprocessor original-feature augmentation interface. Readiness remains provisional: inspect AP/sensitivity/Brier seed SD, sparse-group intervals and attribution stability before prespecifying an augmentation experiment. There is no universal stability cutoff that this run can certify.",
        "Synthetic generation is absent. Validation/test records remain real and untouched by training. Generator provenance and clinical validity require independent verification.",
        "## Limitations", "Single-source, single-split complete-case classification; possible informative censoring; unverified timing of predictors; no verified patient identity beyond exact-record deduplication; small calibration and subgroup samples; no external validation. No clinical deployment claim."])
    report_text = "\n\n".join(report)
    report_path = RUN_DIR / "report.md"
    report_path.write_text(report_text, encoding="utf-8")
    display(Markdown(report_text))
    print("Saved report:", report_path)
else:
    print("Skipped: RUN_FINAL_EVALUATION=False; network development uses training/validation only.")


Skipped: RUN_FINAL_EVALUATION=False; network development uses training/validation only.


## 15. Reuse and validation checklist
Run **Restart Kernel → Run All** after dependency installation. Verify the cohort audit is 3,270/2,821/449, partition membership is complete/disjoint, imported baseline IDs match, six checkpoints exist, thresholds were validation-only, and all requested reports/figures are present. Inspect subgroup flags and SHAP residuals. The notebook intentionally leaves the existing baseline and source CSV unchanged.

To load a checkpoint in a trusted environment: `checkpoint = torch.load(path, map_location=device, weights_only=True)`; construct `MLP(checkpoint['input_dim'], checkpoint['hidden'], checkpoint['dropout']).to(device)` and load `checkpoint['state_dict']`. Apply the saved original-training preprocessor. Maintain its feature mapping and class-1 convention. Artifacts are generated on execution, rather than supplied with fabricated training outputs.


For the fixed-split network-development workflow, leave `RUN_FINAL_EVALUATION=False`: only checkpoints, histories, development metrics, training diagnostics and `network_development.md` are expected. Six selected checkpoints and eighteen candidate histories are produced with the default three candidates/three seeds/two weighting strategies. Full assessment still requires the separately enabled final evaluation.
